# Example 4: Event Detection

Derive maritime events from raw AIS position data.

Neptune detects four types of maritime events from raw AIS positions:
port calls, EEZ crossings, vessel encounters, and loitering.

Each event includes confidence scores, timestamps, and provenance
tracing back to the source data.

## Prerequisites

```bash
pip install neptune-ais[geo,notebooks]
```

**Data requirement:** This example requires downloaded position data. Run
[Example 02](02_archival_ingest.ipynb) first to populate the local store,
or point `cache_dir` to an existing data directory.

## Imports

In [ ]:
from neptune_ais.api import Neptune
from neptune_ais.derive.events import (
    detect_port_calls,
    detect_eez_crossings,
    detect_encounters,
    detect_loitering,
    PortCallConfig,
)

## Load Positions

Start by loading position data from the local store. Events are derived
from these raw positions using heuristic detectors.

If you haven't downloaded data yet, `.download()` will fetch it first.
If data is already cached from [Example 02](02_archival_ingest.ipynb), this is a no-op.

In [ ]:
n = Neptune("2024-06-15", sources=["noaa"], cache_dir="/tmp/neptune_demo")
n.download()
positions = n.positions().collect()

## Detect Port Calls

Port call detection identifies sustained low-speed presence within a port boundary.
It requires a `port_regions` GeoSeries from the `BoundaryRegistry`.

In [ ]:
# port_calls = detect_port_calls(positions, port_regions)
# print(f"Port calls detected: {len(port_calls)}")

## Query Events via the API

The `.events()` method is a higher-level interface that runs detection
and returns results as a LazyFrame. Use `min_confidence` to filter by
the detector's confidence score.

In [ ]:
events_lf = n.events(kind="port_call", min_confidence=0.7)
events = events_lf.collect()
print(f"High-confidence port calls: {len(events)}")

## Event Schema

Every event record follows a canonical schema:

| Field | Description |
|---|---|
| `event_id` | Deterministic SHA-1 hash for deduplication |
| `event_type` | `port_call`, `eez_crossing`, `encounter`, `loitering` |
| `mmsi` | Vessel identifier |
| `start_time` | Event start timestamp |
| `end_time` | Event end timestamp (if applicable) |
| `lat`, `lon` | Event location |
| `confidence_score` | 0.0–1.0 score from the detector |
| `provenance` | `source:detector/version[upstream]` token |

### Confidence Bands

| Band | Range | Meaning |
|---|---|---|
| **HIGH** | >= 0.7 | Strong signal, high certainty |
| **MEDIUM** | 0.3–0.7 | Probable event, may need review |
| **LOW** | < 0.3 | Weak signal, likely noise |

## Next Steps

- **[05 — Streaming Pipeline](05_streaming_pipeline.ipynb)**: Connect to live AIS feeds and detect events in real time
- See [HEURISTICS.md](../HEURISTICS.md) for detection assumptions and known limitations